In [1]:
from chembl_webresource_client.new_client import new_client

target = new_client.target
activity = new_client.activity

results = target.filter(pref_name__icontains="toll-like receptor",
                        organism="Homo sapiens")

for t in results:
    tid = t["target_chembl_id"]
    n = len(activity.filter(target_chembl_id=tid,
                            pchembl_value__isnull=False))
    print(f"{n:6d}   {tid}   {t['pref_name']}")

   277   CHEMBL5255   Toll-like receptor 4
   334   CHEMBL4163   Toll-like receptor 2
  6831   CHEMBL5804   Toll-like receptor 9
  7632   CHEMBL5805   Toll-like receptor 8
  9107   CHEMBL5936   Toll-like receptor 7
    11   CHEMBL1075113   Toll-like receptor 3
     0   CHEMBL2111471   Toll-like receptor (TLR7/TLR9)
     6   CHEMBL2176839   Toll-like receptor 5
     3   CHEMBL3038512   Toll-like receptor 4/MD-2
     6   CHEMBL3038513   Toll-like receptor 4/MD-2/CD14
     0   CHEMBL3259477   Toll-like receptor 6
     0   CHEMBL3714412   Toll-like receptor 1
   198   CHEMBL3885643   Toll-like receptor 1/2
     0   CHEMBL4106182   Toll-like receptor 3/8/9


In [3]:
import os
import pandas as pd
from chembl_webresource_client.new_client import new_client

os.makedirs("data", exist_ok=True)

activity = new_client.activity

records = activity.filter(
    target_chembl_id="CHEMBL5936",
    pchembl_value__isnull=False
).only([
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "pchembl_value",
    "assay_type",
    "assay_chembl_id",
    "document_chembl_id",
])

df = pd.DataFrame(records)
df.to_csv("data/tlr7_raw.csv", index=False)

print("rows:", len(df))
print("columns:", list(df.columns))
df.head()

rows: 9107
columns: ['assay_chembl_id', 'assay_type', 'canonical_smiles', 'document_chembl_id', 'molecule_chembl_id', 'pchembl_value', 'relation', 'standard_relation', 'standard_type', 'standard_units', 'standard_value', 'type', 'units', 'value']


,assay_chembl_id,assay_type,canonical_smiles,document_chembl_id,molecule_chembl_id,pchembl_value,relation,standard_relation,standard_type,standard_units,standard_value,type,units,value
0,CHEMBL975308,F,CCNCc1nc2c3ccccc3nc(N)c2n1CC(C)(C)O.O=C(O)C(F)...,CHEMBL1153973,CHEMBL470427,4.60,=,=,IC50,nM,25000.0,IC50,uM,25.0
1,CHEMBL975308,F,CCNCc1nc2c3ccccc3ncc2n1CC(C)(C)O.O=C(O)C(F)(F)...,CHEMBL1153973,CHEMBL512499,5.12,=,=,IC50,nM,7500.0,IC50,uM,7.5
2,CHEMBL992619,F,COCCOc1nc(N)c2nc(S)n(Cc3cccc(CP(C)(=O)O)c3)c2n1,CHEMBL1142478,CHEMBL519690,8.00,=,=,EC50,nM,10.0,EC50,nM,10.0
3,CHEMBL992620,F,COCCOc1nc(N)c2nc(S)n(Cc3cccc(CP(C)(=O)O)c3)c2n1,CHEMBL1142478,CHEMBL519690,8.00,=,=,EC50,nM,10.0,EC50,nM,10.0
4,CHEMBL992620,F,Nc1nc(NCc2ccccc2)nc2c1nc(O)n2Cc1cccc(CP(=O)(O)...,CHEMBL1142478,CHEMBL483597,8.00,=,=,EC50,nM,10.0,EC50,nM,10.0


In [4]:
print(df["standard_type"].value_counts().head(10))
print()
print(df["assay_type"].value_counts())
print()
print("unique molecules:", df["molecule_chembl_id"].nunique())
print(df["pchembl_value"].astype(float).describe())

standard_type
IC50    7459
EC50    1647
Kd         1
Name: count, dtype: int64

assay_type
B    8938
F     169
Name: count, dtype: int64

unique molecules: 5412
count    9107.000000
mean        7.779538
std         1.231421
min         4.020000
25%         7.000000
50%         8.100000
75%         8.730000
max        10.150000
Name: pchembl_value, dtype: float64


In [5]:
d = df.copy()
d["pchembl_value"] = d["pchembl_value"].astype(float)

# keep only IC50 (blocking), drop rows without a structure
d = d[d["standard_type"] == "IC50"]
d = d.dropna(subset=["canonical_smiles"])

# one row per molecule, using the median of its measurements
g = (d.groupby("molecule_chembl_id")
       .agg(smiles=("canonical_smiles", "first"),
            pchembl=("pchembl_value", "median"),
            n_meas=("pchembl_value", "size"))
       .reset_index())

print("unique molecules after cleaning:", len(g))
print()
print(g["pchembl"].describe())
print()
for t in [6, 6.5, 7, 7.5, 8, 8.5]:
    n_act = (g["pchembl"] >= t).sum()
    print(f"cut {t}:  active {n_act} ({n_act/len(g)*100:.0f}%)   inactive {len(g)-n_act}")

g.to_csv("data/tlr7_clean.csv", index=False)

unique molecules after cleaning: 4039

count    4039.000000
mean        7.703996
std         1.191637
min         4.080000
25%         6.960000
50%         7.995000
75%         8.620000
max        10.000000
Name: pchembl, dtype: float64

cut 6:  active 3594 (89%)   inactive 445
cut 6.5:  active 3320 (82%)   inactive 719
cut 7:  active 3013 (75%)   inactive 1026
cut 7.5:  active 2580 (64%)   inactive 1459
cut 8:  active 2019 (50%)   inactive 2020
cut 8.5:  active 1226 (30%)   inactive 2813


In [6]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

CUT = 7.0
g["active"] = (g["pchembl"] >= CUT).astype(int)

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

fps, labels, kept = [], [], []
bad = 0
for smi, lab in zip(g["smiles"], g["active"]):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        bad += 1
        continue
    fps.append(gen.GetFingerprintAsNumPy(mol))
    labels.append(lab)
    kept.append(smi)

X = np.array(fps, dtype=np.uint8)
y = np.array(labels)

print("failed to read:", bad)
print("X shape:", X.shape)
print("active:", int(y.sum()), "  inactive:", int((y == 0).sum()))

np.save("data/X.npy", X)
np.save("data/y.npy", y)
g.loc[[Chem.MolFromSmiles(s) is not None for s in g["smiles"]]].to_csv("data/tlr7_labelled.csv", index=False)

failed to read: 0
X shape: (4039, 2048)
active: 3013   inactive: 1026
